# Reflexion Architecture — LangGraph

**Pattern:** generator → critic → conditional edge (retry or done)

```
START → generator → critic ─┬─→ generator (retry if score < 7)
                            └─→ finalize → END
```

**LangGraph advantage:** The loop edge is explicit in the graph — visible, bounded, and observable via `stream_mode`.

In [ ]:
import os, re
from typing import TypedDict, Optional
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"
llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
THRESHOLD = 7; MAX_RETRIES = 2
print("✓ Ready")

In [ ]:
class ReflexionState(TypedDict):
    destination: str; draft: Optional[str]; score: int
    issues: str; attempts: int; final_output: Optional[str]

def generator_node(s):
    prompt = f"Write a travel recommendation for {s['destination']}. Include: top 3 attractions, best time, weather, safety."
    if s["issues"]: prompt += f"\n\nFix these issues: {s['issues']}"
    r = llm.invoke([SystemMessage(content="You are a travel writer."), HumanMessage(content=prompt)])
    print(f"  [generator] attempt {s['attempts']+1}")
    return {"draft": r.content, "attempts": s["attempts"]+1}

def critic_node(s):
    r = llm.invoke([SystemMessage(content="Score 1-10 on: specificity(3pts),weather(2pts),safety(2pts),detail(3pts). Format: SCORE:[n] ISSUES:[list]"),
                    HumanMessage(content=f"Recommendation:\n{s['draft']}\n\nScore:")])
    score_m = re.search(r"SCORE:(\d+)", r.content)
    issues_m = re.search(r"ISSUES:(.+?)(?:\n|$)", r.content)
    score = int(score_m.group(1)) if score_m else 5
    issues = issues_m.group(1).strip() if issues_m else ""
    print(f"  [critic] score {score}/10")
    return {"score": score, "issues": issues}

def finalize(s):
    print(f"  [done] score {s['score']}/10 in {s['attempts']} attempt(s)")
    return {"final_output": s["draft"]}

builder = StateGraph(ReflexionState)
builder.add_node("generator", generator_node)
builder.add_node("critic", critic_node)
builder.add_node("finalize", finalize)
builder.add_edge(START, "generator")
builder.add_edge("generator", "critic")
builder.add_conditional_edges("critic",
    lambda s: "generator" if (s["score"] < THRESHOLD and s["attempts"] <= MAX_RETRIES) else "finalize",
    {"generator": "generator", "finalize": "finalize"})
builder.add_edge("finalize", END)
graph = builder.compile()
try:
    from IPython.display import Image, display; display(Image(graph.get_graph().draw_mermaid_png()))
except: print(graph.get_graph().draw_mermaid())

In [ ]:
result = graph.invoke({"destination":"Tokyo","draft":None,"score":0,"issues":"","attempts":0,"final_output":None})
print(f"Final score: {result['score']}/10 | Attempts: {result['attempts']}")
print(result["final_output"])

In [ ]:
# Stream to watch the loop
print("Streaming reflexion loop:")
for step in graph.stream({"destination":"Tokyo","draft":None,"score":0,"issues":"","attempts":0,"final_output":None}, stream_mode="updates"):
    node = list(step.keys())[0]
    written = list(step[node].keys())
    print(f"  [{node}] wrote: {written}")

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| `critic → generator` loop edge | Loop is explicit — graph shows the feedback cycle |
| `attempts` counter in state | Max retries enforced by state, not code |
| `stream_mode="updates"` shows loops | Every retry iteration is visible |

### LangChain vs LangGraph for Reflexion
| | LangChain | LangGraph |
|---|---|---|
| Loop | `for` loop in Python | Conditional edge |
| Observability | Manual logging | `stream_mode="updates"` |
| Max retries | `range(MAX_RETRIES)` | State counter check |
| Visualization | None | `draw_mermaid_png()` shows loop |

### Next: [CrewAI Reflexion →](../CrewAI/reflexion.ipynb)